In [0]:
%pip install --upgrade mlflow>=3.1.3 databricks-agents>=1.1.0 -q
%restart_python

In [0]:
import mlflow
from databricks import agents


catalog = "workspace"
schema = "feature_model"
index = "docs_chunked_index"
model_name = "imda_knowledge_assistant_with_resources"
endpoint_name = "imda_knowledge_assistant"
uc_model_name = f"{catalog}.{schema}.{model_name}"

experiment_name = f"/Workspace/Shared/imda_knowledge_assistant_with_resources"
mlflow.set_experiment(experiment_name=experiment_name)

In [0]:
from mlflow import MlflowClient


def get_latest_model_version(registered_model_name):
    """
    Get the latest version number of a registered model in MLflow.

    Parameters:
    ----------
    registered_model_name: str
        The name of the registered model in MLflow.

    Returns:
    ----------
    int
        The latest version number of the registered model.
    """
    latest_version = 1
    mlflow_client = MlflowClient()
    for mv in mlflow_client.search_model_versions(f"name='{registered_model_name}'"):
        version_int = int(mv.version)
        if version_int > latest_version:
            latest_version = version_int
    return latest_version

In [0]:
get_latest_model_version(uc_model_name)

In [0]:
# deployment = agents.deploy(
#     model_name=uc_model_name, 
#     model_version=get_latest_model_version(uc_model_name),
#     scale_to_zero=True,
#     endpoint_name=endpoint_name,
#     deploy_feedback_model=True
# )

In [0]:
import databricks.agents
print(databricks.agents.__version__)

import mlflow
print(mlflow.__version__)

In [0]:
from mlflow.deployments import get_deploy_client
import time
from mlflow.deployments.databricks import DatabricksDeploymentClient


deployment_client = get_deploy_client("databricks")

def create_or_update_endpoint_with_aigateway(
    deployment_client: DatabricksDeploymentClient,
    endpoint_name: str,
    endpoint_config: dict,
    ai_gateway_config: dict,
):
    """
    Create or update a Databricks endpoint with updated AI Gateway.

    This function checks if an endpoint with the given name exists. If it does not exist, it creates a new endpoint
    with the specified configuration. If it does exist, it updates the existing endpoint with the new configuration.

    Args:
        deployment_client (DatabricksDeploymentClient): The Databricks deployment client used to interact with the Databricks API.
        endpoint_name (str): The name of the endpoint to create or update.
        endpoint_config (dict): The configuration for the endpoint.
        ai_gateway_config (dict): The configuration for the AI Gateway.

    Raises:
        Exception: If the endpoint creation or update fails.
    """
    full_config = {
        "ai_gateway": ai_gateway_config,
        "config": endpoint_config,
    }

    endpoint_names = [
        endpoint["name"] for endpoint in deployment_client.list_endpoints()
    ]

    if endpoint_name not in endpoint_names:
        print(f"Creating a new endpoint with name: {endpoint_name}...")
        endpoint = deployment_client.create_endpoint(
            name=endpoint_name,
            config=full_config,
        )
        while True:
            endpoint = deployment_client.get_endpoint(endpoint_name)
            print(endpoint["state"])
            if endpoint["state"]["config_update"] == "UPDATE_FAILED":
                print(endpoint["state"])
                raise Exception(f"Endpoint creation failed, check the logs for more details")
            if endpoint["state"]["ready"] == "READY":
                print(endpoint["state"])
                print(f"Successfully deployed endpoint of name: {endpoint_name}")
                break
            time.sleep(10)
    else:
        print(
            f"Endpoint of name: {endpoint_name} already exists. Attempting to update the endpoint"
        )
        endpoint = deployment_client.update_endpoint_config(
            endpoint=endpoint_name, config=endpoint_config
        )
        endpoint = deployment_client.update_endpoint_ai_gateway(
            endpoint=endpoint_name, config=ai_gateway_config
        )

        while True:
            endpoint = deployment_client.get_endpoint(endpoint_name)
            print(endpoint["state"])
            
            if endpoint["state"]["config_update"] == "UPDATE_FAILED":
                print(endpoint["state"])
                raise Exception(f"Endpoint creation failed, check the logs for more details")

            if (
                endpoint["state"]["config_update"] == "NOT_UPDATING"
                and endpoint["state"]["ready"] == "READY"
            ):
                print(endpoint["state"])
                print(f"Successfully updated endpoint of name: {endpoint_name}")
                break
            time.sleep(10)



endpoint_config = {
    "served_entities": [
        {
            "name": f"{model_name}_entity",
            "entity_name": uc_model_name,
            "entity_version": str(get_latest_model_version(uc_model_name)),
            "workload_size": "Small",
            "scale_to_zero_enabled": True,
            "workload_type": "CPU",
        }
    ],
    "traffic_config": {
        "routes": [
            {
                "served_model_name": f"{model_name}_entity",
                "traffic_percentage": 100,
            }
        ]
    },
}

ai_gateway_config = {
    # "inference_table_config": {
    #     "catalog_name": catalog,
    #     "enabled": "true",
    #     "schema_name": schema_output,
    # },
    "usage_tracking_config": {"enabled": "false"},
}




create_or_update_endpoint_with_aigateway(
    deployment_client=deployment_client,
    endpoint_name=endpoint_name,
    endpoint_config=endpoint_config,
    ai_gateway_config=ai_gateway_config,
)
